In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv('/content/NPPE1_Preprocessing1.csv')

In [ ]:
df.head()

,CRIM,ZN,INDUS,POLINDEX,RM,AGE,DIS,HIGHWAYCOUNT,TAX,PTRATIO,IMM,BPL,PRICE,RIVERSIDE
0,1.026769,1.429034,7.851300,1.134216,6.0,42.0,5.251911,5,279.201277,20.689586,398.811960,10.461456,22.991633,NO
1,0.848089,0.255543,6.263434,1.245993,7.0,63.0,4.305546,8,307.444529,17.465398,377.153649,11.619690,24.551055,NO
2,10.925905,0.441022,18.322960,2.824833,8.0,-2.0,2.409495,25,666.492973,20.351601,387.061355,19.366070,15.875346,NO
3,0.559027,1.041175,11.114920,0.794952,6.0,9.0,6.898669,4,305.514181,19.787314,391.778647,6.206820,23.007756,NO
4,0.905063,81.167963,3.673369,1.029030,8.0,20.0,10.246463,1,315.913960,17.360439,395.833166,10.827105,21.503177,NO


In [ ]:
df.shape

(4000, 14)

In [ ]:
df['PRICE'].mean()

np.float64(24.355923220694248)

In [ ]:
# df[df['RM']>=]

In [ ]:
df.query('RM>=5').shape[0]

3953

In [ ]:
# #5
# df['PRICE'].sort(ascending=True)[:11,-2].mean()

In [ ]:
df['PRICE'].nlargest(10).mean()

np.float64(52.36590175716407)

In [ ]:
df['RM'].isnull().sum()

np.int64(0)

In [ ]:
#7
df.query('AGE<0').shape[0]

50

In [ ]:
#8
df['RIVERSIDE'].value_counts()


,count
RIVERSIDE,
NO,3595
YES,317
UNKNOWN,88


In [ ]:
df.query("RIVERSIDE == 'YES' and AGE <= 50").shape[0]

53

In [ ]:
#10
df.query("HIGHWAYCOUNT in [6,7,8]").shape[0]

1211

In [ ]:
def category(n):
  if n<10:
    return 1
  elif 10<=n<20:
    return 2
  elif 20<=n<30:
    return 3
  elif 30<=n<40:
    return 4
  else:
    return 5
df['CATEGORY']=df['PRICE'].apply(category)

In [ ]:
df['CATEGORY'].value_counts()

,count
CATEGORY,
3,2028
2,1158
4,503
5,268
1,43


# 12

In [ ]:
df.loc[df['RM'] < 0, 'RM'] = np.nan


In [ ]:
df.loc[df['RIVERSIDE']=='UNKNOWN','RIVERSIDE']=np.nan

In [ ]:
df.loc[df['AGE']<0,'AGE']=np.nan

In [ ]:
df['AGE'].isnull().sum()

np.int64(50)

In [ ]:
from sklearn.model_selection import train_test_split
train_test_split?

In [ ]:
X=df.drop('PRICE',axis=1)
y=df['PRICE']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)
print(X_train.shape)


(2800, 14)


# 13

In [ ]:
X_test=X_test.drop('CATEGORY',axis=1)
X_train=X_train.drop('CATEGORY',axis=1)


In [ ]:
# from sklearn.preprocessing import OneHotEncoder
# OneHotEncodercoder

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

# Columns that ONLY need MinMaxScaler (no imputation needed)
minmax_only = ['CRIM', 'ZN', 'POLINDEX', 'DIS', 'HIGHWAYCOUNT', 'TAX', 'PTRATIO', 'IMM', 'BPL']

# Multi-step pipelines for columns needing imputation first
rm_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', MinMaxScaler())
])

age_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', MinMaxScaler())
])

riverside_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# Combine everything in a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('rm', rm_pipe, ['RM']),
        ('age', age_pipe, ['AGE']),
        ('riverside', riverside_pipe, ['RIVERSIDE']),
        ('minmax', MinMaxScaler(), minmax_only),
        ('indus', StandardScaler(), ['INDUS'])
    ],
    remainder='passthrough'
).set_output(transform='pandas')


In [ ]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [ ]:
X_train

,rm__RM,age__AGE,riverside__RIVERSIDE_NO,riverside__RIVERSIDE_YES,minmax__CRIM,minmax__ZN,minmax__POLINDEX,minmax__DIS,minmax__HIGHWAYCOUNT,minmax__TAX,minmax__PTRATIO,minmax__IMM,minmax__BPL,indus__INDUS
3336,0.428571,0.860000,1.0,0.0,0.189700,0.004561,0.033842,0.124347,0.192308,0.374853,0.829378,0.989745,0.419050,-0.239852
2540,0.571429,0.640453,0.0,1.0,0.036362,0.202330,0.162887,0.271862,0.115385,0.069331,0.690556,0.976549,0.140763,-0.529958
332,0.142857,0.970000,1.0,0.0,0.531739,0.010602,0.652160,0.085051,0.884615,0.912012,0.792344,0.937695,0.047807,1.194743
852,0.285714,0.960000,1.0,0.0,0.358052,0.009848,0.301953,0.067032,0.192308,0.410736,0.214812,0.658560,0.436328,1.289528
1036,0.714286,0.560000,1.0,0.0,0.110154,0.017579,0.158678,0.325495,0.192308,0.223330,0.658217,0.992762,0.324922,-0.038482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,0.285714,0.860000,0.0,1.0,0.108279,0.006169,0.271625,0.137714,0.153846,0.411110,0.283893,0.860627,0.306643,1.333493
3264,0.428571,0.380000,1.0,0.0,0.038962,0.005680,0.179231,0.490348,0.153846,0.191885,0.709585,0.995029,0.149164,-0.474956
1653,0.285714,0.390000,1.0,0.0,0.231637,0.002989,0.474648,0.281366,0.884615,0.911096,0.775340,0.982969,0.266071,1.114639
2607,0.571429,0.400000,1.0,0.0,0.019910,0.010449,0.248182,0.139128,0.192308,0.388282,0.613336,0.992426,0.361401,-0.105376


In [ ]:
X_train.shape

(2800, 14)

In [ ]:
X_test.mean().mean()

np.float64(0.38617456550975626)

# Model Building

In [ ]:
#1
df=pd.read_csv('/content/NPPE1_ModelBuilding3.csv')
df.shape

(4000, 15)

In [ ]:
x=df.drop('14',axis=1)
y=df['14']
x.shape

(4000, 14)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
X_train.shape

(2800, 14)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import  r2_score
model=Ridge(alpha=10,solver='saga',tol=1e-4,random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print(r2_score(y_test,pred))

0.6613547575262211


In [ ]:
n = model.coef_
max_val = -1
index = 0

for idx, val in enumerate(n):
    if abs(val) > max_val:#here we have given importance to positive value Ai told it had no differen in - or + the highest must be given
        max_val = abs(val)
        index = idx

print(index)


9


In [ ]:
np.argmin(model.coef_)

np.int64(9)

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

param = {
    'penalty': ['l1', 'l2'],
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'tol': [1e-4, 1e-3, 1e-2, 1e-1]
}

model = SGDRegressor(random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param,
    cv=5,
    scoring='neg_mean_absolute_error'
)

# Fit on the preprocessed training data
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SGDRegressor(random_state=42),
             param_grid={'alpha': [1e-05, 0.0001, 0.001, 0.01, 0.1],
                         'penalty': ['l1', 'l2'],
                         'tol': [0.0001, 0.001, 0.01, 0.1]},
             scoring='neg_mean_absolute_error')

In [ ]:
grid.best_params_

{'alpha': 0.001, 'penalty': 'l2', 'tol': 0.0001}

In [ ]:
from sklearn.metrics import  mean_absolute_error
pred=grid.predict(X_test)
print(mean_absolute_error(y_test,pred))

3.8131121797994014


In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('Pca', PCA()),
    ('lasso', Lasso())
])

param = {
    'Pca__n_components': [0.9, 0.95],
    'lasso__alpha': [10, 1, 0.01, 0.001]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# Fit on preprocessed training data
grid.fit(X_train, y_train)

# Predict on preprocessed test data and compute R2 score
y_pred = grid.predict(X_test)
print("Best parameters:", grid.best_params_)
print("Test R2 score:", r2_score(y_test, y_pred))


Best parameters: {'Pca__n_components': 0.95, 'lasso__alpha': 0.01}
Test R2 score: 0.6288625430197575


In [ ]:
grid.best_estimator_

Pipeline(steps=[('Pca', PCA(n_components=0.95)), ('lasso', Lasso(alpha=0.01))])

In [ ]:
pca=grid.best_estimator_.named_steps['Pca']

In [ ]:
pca.explained_variance_

array([1.16350757, 0.15109745, 0.11321585, 0.07019734, 0.03866388,
       0.02770669, 0.02515772])

In [ ]:
from sklearn.preprocessing import  PolynomialFeatures
model=Pipeline(
    [
        ('poly', PolynomialFeatures(degree=2,interaction_only=False)),
        ('laso',Lasso(alpha=1,warm_start=True,random_state=0))
    ]
)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print(r2_score(y_test,pred))

0.157678032410551
